In [ ]:
import os
import re
import json
import time
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from pricer.items import Item
from pricer.evaluator import evaluate, Tester

load_dotenv(override=True)
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(hf_token, add_to_git_credential=True)

openai = OpenAI()


In [ ]:
LITE_MODE = True  
USERNAME = "ed-donner"
dataset_name = f"{USERNAME}/items_lite" if LITE_MODE else f"{USERNAME}/items_full"

train, val, test = Item.from_hub(dataset_name)
print(f"Train: {len(train):,}  |  Val: {len(val):,}  |  Test: {len(test):,}")
print(f"Example: {train[0].title[:50]}... → ${train[0].price:.2f}")

In [ ]:
SYSTEM_PROMPT = (
    "You are a product pricing expert. "
    "Given a product description, estimate its fair market price in USD. "
    "Reply with only the price in the format $X.XX (e.g. $29.99). No explanation."
)

def messages_for(item: Item):
    """Training/validation: user + assistant completion."""
    user_content = f"Estimate the price of this product.\n\n{item.summary}"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": f"${item.price:.2f}"},
    ]

def test_messages_for(item: Item):
    """Inference: system + user only (no assistant)."""
    user_content = f"Estimate the price of this product.\n\n{item.summary}"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]

print("Example training messages:")
print(json.dumps(messages_for(train[0]), indent=2)[:600], "...")

In [ ]:
N_TRAIN = 300
N_VAL = 80

fine_tune_train = train[:N_TRAIN]
fine_tune_validation = val[:N_VAL]
print(f"Fine-tune train: {len(fine_tune_train)}  |  Fine-tune val: {len(fine_tune_validation)}")

In [ ]:
def make_jsonl(items):
    lines = []
    for item in items:
        messages = messages_for(item)
        lines.append(json.dumps({"messages": messages}))
    return "\n".join(lines)

def write_jsonl(items, path: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(make_jsonl(items))

JSONL_DIR = Path("jsonl")
train_path = JSONL_DIR / "fine_tune_train.jsonl"
val_path = JSONL_DIR / "fine_tune_validation.jsonl"
write_jsonl(fine_tune_train, train_path)
write_jsonl(fine_tune_validation, val_path)
print(f"Wrote {train_path} ({len(fine_tune_train)} examples), {val_path} ({len(fine_tune_validation)} examples)")

def upload_with_retry(file_path, purpose="fine-tune", max_retries=3):
    from openai import InternalServerError
    for attempt in range(1, max_retries + 1):
        try:
            with open(file_path, "rb") as f:
                return openai.files.create(file=f, purpose=purpose)
        except InternalServerError as e:
            if attempt == max_retries:
                raise
            wait = 60 * attempt
            print(f"500 on upload (attempt {attempt}/{max_retries}). Retrying in {wait}s...")
            time.sleep(wait)
    return None

if not openai:
    train_file = None
    validation_file = None
    print("Skipped (no OPENAI_API_KEY).")
else:
    train_file = upload_with_retry(train_path)
    validation_file = upload_with_retry(val_path)
    print(f"Uploaded train file: {train_file.id}, validation file: {validation_file.id}")

In [ ]:
BASE_MODEL = "gpt-4.1-nano-2025-04-14"


EXPERIMENTS = {
    "baseline": {"n_epochs": 1, "batch_size": 1},
    "more_epochs": {"n_epochs": 2, "batch_size": 1},
    "larger_batch": {"n_epochs": 1, "batch_size": 2},
    "conservative": {"n_epochs": 2, "batch_size": 1, "learning_rate_multiplier": 0.5},
}

CHOSEN = "more_epochs" 
hyperparams = EXPERIMENTS[CHOSEN]
print(f"Using experiment: {CHOSEN}  →  {hyperparams}")

if not openai or not train_file:
    job_id = None
    print("Skipped (no OpenAI client or uploaded files).")
else:
    job = openai.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=validation_file.id,
        model=BASE_MODEL,
        seed=42,
        hyperparameters=hyperparams,
        suffix=f"pricer_{CHOSEN}",
    )
    job_id = job.id
    print(f"Fine-tuning job created: {job_id}")

In [ ]:
job_id = globals().get("job_id", None)
fine_tuned_model_name = None
if job_id and openai:
    while True:
        job = openai.fine_tuning.jobs.retrieve(job_id)
        status = job.status
        print(status, end=" ")
        if status == "succeeded":
            fine_tuned_model_name = job.fine_tuned_model
            print(f"\nDone. Model: {fine_tuned_model_name}")
            break
        if status == "failed":
            print(f"\nJob failed: {getattr(job, 'error', None)}")
            raise RuntimeError("Fine-tuning job failed")
        time.sleep(30)
else:
    print("Skipped (no job_id).")

In [ ]:
EVAL_SIZE = 200 

def fine_tuned_pricer(item: Item):
    if not openai or not fine_tuned_model_name:
        return "0.00"
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=16,
    )
    return (response.choices[0].message.content or "").strip()

predictor = fine_tuned_pricer
evaluate(predictor, test, size=EVAL_SIZE)